In [ ]:
# Lab type: extend
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Query Rewriting: HyDE, Multi-Query, and Decomposition
# Task: A baseline dense retriever is provided, along with canned LLM
# rewrite outputs (no API key needed). Extend the pipeline with multi-query
# fusion and decomposition, and measure what each buys.

# Lab: Extending a Retriever with Query Rewriting

The rewrites an LLM would produce are supplied as canned strings so the lab runs without API calls — in production these come from a versioned rewrite prompt.

**Outputs are cleared.** Run every cell top to bottom.

## Setup: baseline retriever

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

In [ ]:
def rrf_fuse(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def recall_at_k(search_fn, k=3):
    hits = sum(1 for q, rel in EVAL_SET
               if set(search_fn(q)[:k]) & rel)
    return hits / len(EVAL_SET)

baseline = lambda q: [d for d, _ in dense_search(q, k=5)]
print(f"Baseline dense recall@3: {recall_at_k(baseline):.2f}")

## Canned LLM rewrites

For three ambiguous eval queries, here is what a rewrite prompt returned.

In [ ]:
MULTI_QUERY_REWRITES = {
    "how do I get my money back": [
        "refund policy",
        "request a refund for a subscription",
        "cancel my plan and get reimbursed",
        "billing refund window",
    ],
    "how long is event data kept": [
        "data retention period",
        "how many months is analytics data stored",
        "retention window configuration",
    ],
    "response time for priority support": [
        "priority support SLA",
        "support tier response times",
        "how fast does support reply on paid plans",
    ],
}

COMPOUND_QUESTION = (
    "which plan should we pick if we need SSO and priority support "
    "but only have budget for 20 seats"
)
DECOMPOSITION = [
    "which plans include SSO",
    "which plans include priority support",
    "per seat pricing for each plan",
]

## Extension 1: multi-query fusion

Implement `multi_query_search(query)`: retrieve for the original query *and* its rewrites (when available), fuse with RRF, and return ranked doc IDs. Then compare recall@3 against the baseline.

In [ ]:
def multi_query_search(query, k_per_arm=5):
    # TODO: build the list of queries: the ORIGINAL plus any rewrites
    # TODO: run dense_search for each, collect doc-id rankings
    # TODO: return rrf_fuse(rankings)
    pass


<details>
<summary>🔑 Reveal model answer — Extension 1</summary>

```python
def multi_query_search(query, k_per_arm=5):
    queries = [query] + MULTI_QUERY_REWRITES.get(query, [])
    rankings = [[d for d, _ in dense_search(q, k=k_per_arm)]
                for q in queries]
    return rrf_fuse(rankings)

print(f"Multi-query recall@3: {recall_at_k(multi_query_search):.2f}")
```

Keeping the original query in the list is not optional: a rewrite that drifts from the user's intent would otherwise be able to push the right documents out entirely.

</details>

## Extension 2: decomposition

Implement `decomposed_search(sub_questions)`: retrieve top-2 documents per sub-question and return them *grouped by sub-question* (a dict), not fused — generation needs to know which evidence answers which part.

In [ ]:
def decomposed_search(sub_questions, k=2):
    # TODO: return {sub_question: [doc_id, ...]} for each sub-question
    pass

# When done: decomposed_search(DECOMPOSITION)


<details>
<summary>🔑 Reveal model answer — Extension 2</summary>

```python
def decomposed_search(sub_questions, k=2):
    return {sq: [d for d, _ in dense_search(sq, k=k)]
            for sq in sub_questions}

for sq, docs in decomposed_search(DECOMPOSITION).items():
    print(f"{sq!r:.55} -> {docs}")
```

Compare with `dense_search(COMPOUND_QUESTION, k=3)`: the single-shot query averages three information needs into one vector and typically misses at least one of `sso-policy` / `priority-support` / `seat-pricing`; the decomposed version retrieves each fact independently.

</details>

## Extension 3: the cost column

Each rewrite pattern adds LLM calls and extra retrievals. For each of baseline, multi-query (assume 1 rewrite call + N retrievals) and decomposition (1 call + M retrievals), tally: LLM calls before retrieval, retrieval operations, and whether latency is added *before* the first retrieval. Write your table in the markdown cell below.

*(Your cost table here.)*

<details>
<summary>🔑 Reveal model answer — Extension 3</summary>

| pipeline | LLM calls pre-retrieval | retrievals | latency before 1st retrieval |
|---|---|---|---|
| baseline | 0 | 1 | none |
| multi-query | 1 | 1 + #rewrites (parallelisable) | one LLM round-trip |
| decomposition | 1 | #sub-questions | one LLM round-trip |

The rewrite call usually costs more latency than retrieval itself — which is why rewriting is applied selectively (ambiguous/compound traffic), not unconditionally.

</details>

## Summary

1. Multi-query fuses the original plus rewrites with _______.
2. Decomposition returns evidence _______ by sub-question, not fused.
3. Every rewrite pattern inserts an _______ call in front of retrieval — cost, latency, and stochasticity.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **RRF (reciprocal rank fusion)**
2. **grouped**
3. **LLM**

</details>